In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
LEARNING_RATE = 1e-3
EPOCHS = 10

In [3]:
model = torchvision.models.resnet18()

num_classes = 4
model.fc = nn.Linear(model.fc.in_features, 4)

model = model.to(device=device)

In [7]:
class CWRUDataset(Dataset):
    def __init__(self, path):
        self.dataset = torch.load(path)
        self.data = self.dataset['data']
        self.label = self.dataset['label']
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        image = self.data[index] / 255.0
        label = self.label[index].long()

        return image, label

In [8]:
train_dataset = CWRUDataset(os.path.join(root_path, "train.pt"))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = CWRUDataset(os.path.join(root_path, "test.pt"))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

val_dataset = CWRUDataset(os.path.join(root_path, "val.pt"))
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

In [17]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [18]:
def train_one_epoch(epoch_index, tb_writer):
    running_loss = 0.
    last_loss = 0.

    for i, data in enumerate(train_loader):
        inputs, labels = data

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = loss_fn(outputs, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        if i % 1000 == 999:
            last_loss = running_loss / 1000 # loss per batch
            print("batch {} loss {}".format(i + 1, last_loss))
            tb_x = epoch_index * len(train_loader) + i + 1
            tb_writer.add_scalar('Loss/train', last_loss, tb_x)
            running_loss = 0.
            
    return last_loss